In [ ]:
!pip install -q gradio opencv-python-headless tensorflow

In [ ]:
import tensorflow as tf
import cv2
import numpy as np
import gradio as gr
from PIL import Image

# Load pretrained EfficientNetB0
model = tf.keras.applications.EfficientNetB0(
    weights="imagenet",
    include_top=True
)

print("EfficientNetB0 loaded successfully!")
print("TensorFlow version:", tf.__version__)

21834768/21834768 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
EfficientNetB0 loaded successfully!
TensorFlow version: 2.21.0


In [ ]:
def process_image(image):
    if image is None:
        return "Please upload an image."

    image = image.convert("RGB")
    image_array = np.array(image)

    # Resize image for EfficientNetB0
    resized = cv2.resize(image_array, (224, 224))

    # Prepare input
    x = np.expand_dims(resized.astype(np.float32), axis=0)

    # Generate a general image classification prediction
    predictions = model.predict(x, verbose=0)
    decoded = tf.keras.applications.efficientnet.decode_predictions(
        predictions, top=3
    )[0]

    result = "Image processed successfully.\n\n"
    result += "General image classifications:\n"

    for _, label, score in decoded:
        result += f"{label}: {score * 100:.2f}%\n"

    result += (
        "\nNote: These are general image classifications, "
        "NOT real/fake predictions."
    )

    return result


def process_video(video):
    if video is None:
        return "Please upload a video."

    cap = cv2.VideoCapture(video)

    if not cap.isOpened():
        return "Unable to open the video."

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    sampled_frames = []

    # Sample up to 10 frames
    if total_frames > 0:
        indices = np.linspace(
            0, total_frames - 1,
            min(10, total_frames),
            dtype=int
        )

        for index in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
            ret, frame = cap.read()

            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                sampled_frames.append(frame)

    cap.release()

    return (
        f"Video processed successfully!\n"
        f"Total frames: {total_frames}\n"
        f"FPS: {fps:.2f}\n"
        f"Frames sampled: {len(sampled_frames)}\n\n"
        "Note: This prototype has not classified the video "
        "as Real or Deepfake. A trained detection model is required."
    )


print("Image and video functions created!")

Image and video functions created!


In [ ]:
with gr.Blocks(title="Deepfake Detection System") as demo:

    gr.Markdown("# Deepfake Detection System")
    gr.Markdown("Image and Video Analysis Prototype")

    with gr.Tab("Image Analysis"):
        image_input = gr.Image(type="pil", label="Upload Image")
        image_button = gr.Button("Analyze Image")
        image_output = gr.Textbox(label="Analysis Result", lines=8)

        image_button.click(
            fn=process_image,
            inputs=image_input,
            outputs=image_output
        )

    with gr.Tab("Video Analysis"):
        video_input = gr.Video(label="Upload Video")
        video_button = gr.Button("Analyze Video")
        video_output = gr.Textbox(label="Analysis Result", lines=8)

        video_button.click(
            fn=process_video,
            inputs=video_input,
            outputs=video_output
        )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8b7115086202231f4c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
